### YOLOv11x Transfer Learning from Fashionpedia Dataset

#### Libraries

In [1]:
import os
from tqdm.notebook import tqdm
import numpy as np
from datasets import load_dataset
from ultralytics import YOLO

#### Dataset

In [2]:
#Download Dataset
ds = load_dataset("detection-datasets/fashionpedia")

In [3]:
#Explore Dataset
class_names = ds["train"].features["objects"].feature["category"].names
num_classes = len(class_names)
print(f"Found {num_classes} classes in Fashionpedia dataset")

Found 46 classes in Fashionpedia dataset


In [4]:
#List Classes
ds['train'].features['objects'].feature['category'].names

['shirt, blouse',
 'top, t-shirt, sweatshirt',
 'sweater',
 'cardigan',
 'jacket',
 'vest',
 'pants',
 'shorts',
 'skirt',
 'coat',
 'dress',
 'jumpsuit',
 'cape',
 'glasses',
 'hat',
 'headband, head covering, hair accessory',
 'tie',
 'glove',
 'watch',
 'belt',
 'leg warmer',
 'tights, stockings',
 'sock',
 'shoe',
 'bag, wallet',
 'scarf',
 'umbrella',
 'hood',
 'collar',
 'lapel',
 'epaulette',
 'sleeve',
 'pocket',
 'neckline',
 'buckle',
 'zipper',
 'applique',
 'bead',
 'bow',
 'flower',
 'fringe',
 'ribbon',
 'rivet',
 'ruffle',
 'sequin',
 'tassel']

In [5]:
#Test filter function
ds['train'].filter(lambda x: 23 in x['objects']['category'])

Dataset({
    features: ['image_id', 'image', 'width', 'height', 'objects'],
    num_rows: 23954
})

In [ ]:
#List number of instances per class
for idx, class_name in enumerate(ds['train'].features['objects'].feature['category'].names):
    count = ds['train'].filter(lambda x: idx in x['objects']['category']).num_rows
    print(f"Class {class_name}: {count} training images")

Class shirt, blouse: 6115 instances
Class top, t-shirt, sweatshirt: 16176 instances
Class sweater: 1485 instances
Class cardigan: 1101 instances
Class jacket: 7742 instances
Class vest: 717 instances
Class pants: 12347 instances
Class shorts: 2745 instances
Class skirt: 5034 instances
Class coat: 3085 instances
Class dress: 18666 instances
Class jumpsuit: 922 instances
Class cape: 152 instances
Class glasses: 4848 instances
Class hat: 2514 instances
Class headband, head covering, hair accessory: 2984 instances
Class tie: 1455 instances
Class glove: 757 instances
Class watch: 3373 instances
Class belt: 6667 instances
Class leg warmer: 61 instances
Class tights, stockings: 2202 instances
Class sock: 1445 instances
Class shoe: 23954 instances
Class bag, wallet: 6907 instances
Class scarf: 1362 instances
Class umbrella: 134 instances
Class hood: 1216 instances
Class collar: 9801 instances
Class lapel: 5885 instances
Class epaulette: 507 instances
Class sleeve: 29465 instances
Class pocket:

In [ ]:
#Remove from dataset classes with less than 700 images
min_instances = 700
#Loop through all classes and keep only classes with at least 700 image rows/images
ds_filtered = ds.filter(lambda x: any(ds['train'].filter(lambda y: cat in y['objects']['category']).num_rows >= min_instances for cat in x['objects']['category']))
  

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45623 [00:00<?, ? examples/s]

In [59]:
#First training example
[x not in (23, 33) for x in ds["train"][0]['objects']['category']]

[False, False, False, True]

#### Convert COCO to YOLO

In [ ]:
# Create directory structure
output_dir = "/home/tommytang111/data/fashionpedia/yolo_format"
os.makedirs(f"{output_dir}/images/train", exist_ok=True)
os.makedirs(f"{output_dir}/images/val", exist_ok=True)  
os.makedirs(f"{output_dir}/labels/train", exist_ok=True)
os.makedirs(f"{output_dir}/labels/val", exist_ok=True)

In [10]:
def coco_to_yolo_bbox(bbox, img_width, img_height):
    """Convert COCO format [x_min, y_min, width, height] to YOLO format [x_center, y_center, width, height] (normalized)"""
    x_min, y_min, width, height = bbox
    
    # Handle edge cases with invalid bounding boxes
    if width <= 0 or height <= 0:
        return None
        
    # Convert to YOLO format (normalized)
    x_center = (x_min + width / 2) / img_width
    y_center = (y_min + height / 2) / img_height
    width = width / img_width
    height = height / img_height
    
    # Ensure values are in valid range [0, 1]
    if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None
        
    return [x_center, y_center, width, height]

In [26]:
# Process each split
for split in ["train", "val"]:
    yolo_split = "train" if split == "train" else "val"
    print(f"Processing {split} split...")
    
    for i, item in enumerate(tqdm.tqdm(ds[split])):
        # Get image
        img = item["image"]
        img_width, img_height = img.size
        
        # Create unique filename based on index
        filename = f"{i:06d}"
        
        # Save image
        img_path = f"{output_dir}/images/{yolo_split}/{filename}.jpg"
        img.save(img_path)
        
        # Save YOLO label
        label_path = f"{output_dir}/labels/{yolo_split}/{filename}.txt"
        
        with open(label_path, "w") as f:
            # Process each object
            for j in range(len(item["objects"]["bbox"])):
                # Get class ID and bounding box
                class_id = item["objects"]["category"][j]
                bbox = item["objects"]["bbox"][j]
                
                # Convert to YOLO format
                yolo_bbox = coco_to_yolo_bbox(bbox, img_width, img_height)
                
                # Skip invalid bounding boxes
                if yolo_bbox is None:
                    continue
                
                # Write to file: class_id x_center y_center width height
                bbox_str = " ".join([f"{coord:.6f}" for coord in yolo_bbox])
                f.write(f"{class_id} {bbox_str}\n")

Processing train split...


100%|██████████| 45623/45623 [03:42<00:00, 205.43it/s]


Processing val split...


100%|██████████| 1158/1158 [00:06<00:00, 187.36it/s]


In [27]:
# Create data.yaml file
yaml_content = f"""
train: {output_dir}/images/train
val: {output_dir}/images/val

nc: {num_classes}
names: {list(class_names)}
"""

with open(f"{output_dir}/data.yaml", "w") as f:
    f.write(yaml_content)

print(f"Conversion complete. Dataset saved to {output_dir}")
print(f"Created data.yaml with {num_classes} classes")

Conversion complete. Dataset saved to /home/tommytang111/Projects/Vision/data/yolo_format
Created data.yaml with 46 classes


In [6]:
# Examine dataset structure and verify conversion success
!find {output_dir} -type f | wc -l
print("Sample label file content:")
!head -n 3 {output_dir}/labels/train/000000.txt

# Check class distribution
import glob
import re

def count_classes(label_dir):
    class_counts = [0] * num_classes
    for label_file in glob.glob(f"{label_dir}/*.txt"):
        with open(label_file, 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1
    return class_counts

train_class_counts = count_classes(f"{output_dir}/labels/train")
val_class_counts = count_classes(f"{output_dir}/labels/val")

# Display top 10 classes
top_classes = sorted(range(len(train_class_counts)), 
                    key=lambda i: train_class_counts[i], 
                    reverse=True)[:10]

print("\nTop 10 classes by frequency:")
for i, class_id in enumerate(top_classes):
    print(f"{i+1}. {class_names[class_id]}: {train_class_counts[class_id]} train, {val_class_counts[class_id]} val")

93565
Sample label file content:
33 0.719941 0.447266 0.565982 0.343750
10 0.636364 0.600098 0.656891 0.649414

Top 10 classes by frequency:
1. sleeve: 45086 train, 1211 val
2. neckline: 33571 train, 894 val
3. pocket: 19116 train, 388 val
4. dress: 18478 train, 495 val
5. top, t-shirt, sweatshirt: 16083 train, 453 val
6. collar: 9978 train, 215 val
7. jacket: 7694 train, 177 val
8. pants: 7266 train, 218 val
9. zipper: 6520 train, 152 val
10. shirt, blouse: 6056 train, 102 val


#### Training

In [4]:
#Load Model
model = YOLO('yolo11m')

In [19]:
model.model

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   

#### Freeze Layers 1-8

In [5]:
# Optimized Transfer Learning for Highest IoU Results
pytorch_model = model.model

# For highest IoU, freeze backbone + early neck features
freeze_until_layer = 8  # Optimal for IoU performance

frozen_layers = []
trainable_layers = []

for i, (name, param) in enumerate(pytorch_model.named_parameters()):
    layer_num = int(name.split('.')[1]) if 'model.' in name and name.split('.')[1].isdigit() else -1
    
    if layer_num <= freeze_until_layer:
        param.requires_grad = False
        frozen_layers.append(name)
    else:
        param.requires_grad = True
        trainable_layers.append(name)

print(f"Frozen layers (0-{freeze_until_layer}): {len(frozen_layers)} parameters")
print(f"Trainable layers ({freeze_until_layer+1}-23): {len(trainable_layers)} parameters")

# Verify the freeze configuration
frozen_params = sum(1 for param in pytorch_model.parameters() if not param.requires_grad)
total_params = sum(1 for param in pytorch_model.parameters())
trainable_params = total_params - frozen_params

print(f"\nParameter Summary:")
print(f"Total parameters: {total_params}")
print(f"Frozen parameters: {frozen_params}")
print(f"Trainable parameters: {trainable_params}")
print(f"Percentage trainable: {trainable_params/total_params*100:.1f}%")

print(f"\nFrozen Layers (0-8): Robust feature extraction")
print("- Layers 0-6: Backbone (edges, textures, basic patterns)")
print("- Layers 7-8: Early neck (complex patterns, initial fusion)")

print(f"\nTrainable Layers (9-23): IoU optimization")
print("- Layer 9: SPPF (spatial pyramid pooling)")
print("- Layer 10: C2PSA (attention mechanism)")
print("- Layers 11-16: FPN (multi-scale feature fusion)")
print("- Layers 17-23: PAN + Detection head (precise localization)")

Frozen layers (0-8): 123 parameters
Trainable layers (9-23): 208 parameters

Parameter Summary:
Total parameters: 331
Frozen parameters: 123
Trainable parameters: 208
Percentage trainable: 62.8%

Frozen Layers (0-8): Robust feature extraction
- Layers 0-6: Backbone (edges, textures, basic patterns)
- Layers 7-8: Early neck (complex patterns, initial fusion)

Trainable Layers (9-23): IoU optimization
- Layer 9: SPPF (spatial pyramid pooling)
- Layer 10: C2PSA (attention mechanism)
- Layers 11-16: FPN (multi-scale feature fusion)
- Layers 17-23: PAN + Detection head (precise localization)


#### Run training

In [6]:
# Train YOLOv11m on Fashionpedia
results = model.train(
    data=f'{output_dir}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,  
    device='cuda:0',
    pretrained=True,
    workers=4,
    patience=15,
    lr0=0.002, 
    cos_lr=True,
    weight_decay=0.0005,
    dropout=0.0,
    warmup_epochs=3,
    optimizer='AdamW',
    warmup_bias_lr=0.1,
    warmup_momentum=0.8,
    name='yolov11m-fashionpedia-v1'
)

print(f"Training complete. Best model saved at: {results.best}")

New https://pypi.org/project/ultralytics/8.3.148 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.147 🚀 Python-3.10.16 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/tommytang111/Vision/data/yolo_format/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov11m-fashion

train: Scanning /home/tommytang111/Vision/data/yolo_format/labels/train.cache... 45623 images, 206 backgrounds, 0 corrupt: 100%|██████████| 45623/45623 [00:00<?, ?it/s]


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1915.4±1455.9 MB/s, size: 70.7 KB)


val: Scanning /home/tommytang111/Vision/data/yolo_format/labels/val.cache... 1158 images, 14 backgrounds, 0 corrupt: 100%|██████████| 1158/1158 [00:00<?, ?it/s]


Plotting labels to runs/detect/yolov11m-fashionpedia-v14/labels.jpg... 
optimizer: AdamW(lr=0.002, momentum=0.937) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.0005), 112 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to runs/detect/yolov11m-fashionpedia-v14
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all       1158       6111      0.572      0.433      0.426      0.358



50 epochs completed in 17.406 hours.
Optimizer stripped from runs/detect/yolov11m-fashionpedia-v14/weights/last.pt, 40.6MB
Optimizer stripped from runs/detect/yolov11m-fashionpedia-v14/weights/best.pt, 40.6MB

Validating runs/detect/yolov11m-fashionpedia-v14/weights/best.pt...
Ultralytics 8.3.147 🚀 Python-3.10.16 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
YOLO11m summary (fused): 125 layers, 20,065,498 parameters, 0 gradients, 67.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 73/73 [00:13<00:00,  5.59it/s]


                   all       1158       6111      0.572      0.433      0.426      0.358
         shirt, blouse        102        102      0.461      0.461      0.496      0.411
top, t-shirt, sweatshirt        441        453      0.528      0.728      0.666      0.582
               sweater         21         21      0.262      0.238      0.245      0.211
              cardigan         10         10     0.0905        0.1     0.0849     0.0829
                jacket        173        177      0.573      0.785      0.759      0.691
                  vest         20         21      0.428      0.238      0.282      0.228
                 pants        218        218      0.624      0.917      0.873      0.828
                shorts         62         62      0.467      0.708      0.668      0.619
                 skirt        135        135      0.519       0.63      0.634      0.613
                  coat        101        101      0.574      0.812      0.758      0.678
                 dr

AttributeError: 'DetMetrics' object has no attribute 'best'. See valid attributes below.

    Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

    Attributes:
        save_dir (Path): A path to the directory where the output plots will be saved.
        plot (bool): A flag that indicates whether to plot precision-recall curves for each class.
        names (Dict[int, str]): A dictionary of class names.
        box (Metric): An instance of the Metric class for storing detection results.
        speed (Dict[str, float]): A dictionary for storing execution times of different parts of the detection process.
        task (str): The task type, set to 'detect'.
    